# v2模型 + Benchmark回测策略

使用v2模型的预测，但回测策略与benchmark完全一致。

In [ ]:
import sys, json
from pathlib import Path
import numpy as np
import pandas as pd
import torch, torch.nn as nn
from sklearn.preprocessing import StandardScaler
import warnings; warnings.filterwarnings('ignore')
PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'pyproject.toml').exists()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError('请从 Quant 项目目录或其子目录启动 Jupyter')
sys.path.insert(0, str(PROJECT_ROOT))
from strategies.futures.alstm.data import load_minute_bars
from engine.backtest.futures_backtest import StrategyRuleConfig, run_backtest
print("完成")

In [ ]:
# 读取benchmark的回测结果
import pandas as pd
bm = pd.read_csv(PROJECT_ROOT / "artifacts/futures_alstm/if_alstm_rolling_3state_intra_2025/pv3_two_stage_trade_direction_model_b_compare/alstm3_retrain_ab_split_raw_634d/window_summary.csv")
bm_nb = bm[bm['strategy'] == 'a_alstm_b_binary_nonflat_raw']
print("Benchmark (ALSTM nonflat_binary):")
print(f"{'窗口':<12} {'收益':<12} {'夏普':<12} {'回撤':<12} {'交易次数':<12}")
print("-" * 60)
for _, r in bm_nb.iterrows():
    print(f"{r['window_id']:<12} {r['total_return']*100:>+.2f}%{'':<6} {r.get('annualized_sharpe',0):>+.2f}{'':<6} {r['max_drawdown']*100:>+.2f}%{'':<6} {int(r['trade_count']):>5}")

In [ ]:
# 加载数据（使用CSV，与v2训练时一致）
raw_csv = load_minute_bars("IF")
raw_csv['datetime'] = pd.to_datetime(raw_csv['datetime'])
raw_csv['returns_1min'] = raw_csv['vwap'].pct_change()
raw_csv['returns_5min'] = raw_csv['vwap'].pct_change(5)
raw_csv['returns_15min'] = raw_csv['vwap'].pct_change(15)
raw_csv['volatility_5'] = raw_csv['returns_1min'].rolling(5).std()
raw_csv['volatility_10'] = raw_csv['returns_1min'].rolling(10).std()
raw_csv['volatility_20'] = raw_csv['returns_1min'].rolling(20).std()
raw_csv['momentum_5'] = raw_csv['vwap'] / raw_csv['vwap'].shift(5) - 1
raw_csv['momentum_10'] = raw_csv['vwap'] / raw_csv['vwap'].shift(10) - 1
raw_csv['momentum_20'] = raw_csv['vwap'] / raw_csv['vwap'].shift(20) - 1
raw_csv['sma_5'] = raw_csv['vwap'].rolling(5).mean()
raw_csv['sma_10'] = raw_csv['vwap'].rolling(10).mean()
raw_csv['sma_20'] = raw_csv['vwap'].rolling(20).mean()
raw_csv['volume_ratio_5'] = raw_csv['volume'] / (raw_csv['volume'].rolling(5).mean() + 1e-12)
raw_csv['volume_ratio_10'] = raw_csv['volume'] / (raw_csv['volume'].rolling(10).mean() + 1e-12)
raw_csv['atr_5'] = (raw_csv['high'] / raw_csv['low'] - 1).rolling(5).mean()
raw_csv['atr_10'] = (raw_csv['high'] / raw_csv['low'] - 1).rolling(10).mean()
raw_csv['bb_position_20'] = (raw_csv['vwap'] - raw_csv['sma_20']) / (2 * raw_csv['vwap'].rolling(20).std() + 1e-12)
raw_csv['rsi_20'] = 100 * (raw_csv['returns_1min'].clip(lower=0).rolling(20).mean()) / (raw_csv['returns_1min'].abs().rolling(20).mean() + 1e-12)
raw_csv['close_open_ratio'] = raw_csv['close'] / (raw_csv['open'] + 1e-12)
raw_csv['high_low_ratio'] = raw_csv['high'] / (raw_csv['low'] + 1e-12)
raw_csv['price_position'] = (raw_csv['vwap'] - raw_csv['low']) / (raw_csv['high'] - raw_csv['low'] + 1e-12)
base_features = ['returns_1min','returns_5min','returns_15min','volatility_5','volatility_10','volatility_20','momentum_5','momentum_10','momentum_20','sma_5','sma_10','sma_20','volume_ratio_5','volume_ratio_10','atr_5','atr_10','bb_position_20','rsi_20','close_open_ratio','high_low_ratio','price_position']

# L2 tick因子
frames = []
for month in range(7, 13):
    try:
        df = pd.read_parquet(PROJECT_ROOT / f"data_lake/future/market_data/tick_l2/year=2025/month={month:02d}.parquet")
        cffex = df[df['exchange'] == 'CFFEX']
        if_df = cffex[cffex['code'].str.startswith('IF')]
        if len(if_df) > 0: frames.append(if_df)
    except: pass
tick_df = pd.concat(frames).sort_values('timestamp').reset_index(drop=True)
tick_df['timestamp'] = pd.to_datetime(tick_df['timestamp'])
bid_vol = tick_df[['bid_vol_1','bid_vol_2','bid_vol_3','bid_vol_4','bid_vol_5']].sum(axis=1)
ask_vol = tick_df[['ask_vol_1','ask_vol_2','ask_vol_3','ask_vol_4','ask_vol_5']].sum(axis=1)
tick_df['l2_spread'] = (tick_df['ask_px_1'] - tick_df['bid_px_1']) / tick_df['last_price']
tick_df['l2_imbalance'] = (bid_vol - ask_vol) / (bid_vol + ask_vol + 1e-12)
tick_df['l2_mid_price'] = (tick_df['ask_px_1'] + tick_df['bid_px_1']) / 2
tick_df['l2_volume_change'] = tick_df['volume'].diff()
tick_df['l2_oi_change'] = tick_df['open_interest'].diff()
tick_df['l2_price_momentum'] = tick_df['last_price'].pct_change()
tick_df['minute'] = tick_df['timestamp'].dt.floor('1min')
l2_cols = ['l2_spread','l2_imbalance','l2_mid_price','l2_volume_change','l2_oi_change','l2_price_momentum']
tick_minute = tick_df.groupby('minute')[l2_cols].mean().reset_index().rename(columns={'minute':'datetime'})
tick_minute['datetime'] = pd.to_datetime(tick_minute['datetime'])

all_features = base_features + l2_cols
merged = raw_csv.merge(tick_minute, on='datetime', how='left')
merged[l2_cols] = merged[l2_cols].fillna(0)
merged = merged.dropna(subset=base_features)
print(f"数据: {len(merged):,} 行, 特征: {len(all_features)}")

In [ ]:
# v2模型
class AttnLSTM(nn.Module):
    def __init__(self, nf, nc):
        super().__init__()
        self.bn = nn.BatchNorm1d(nf)
        self.lstm = nn.LSTM(nf, 64, 2, batch_first=True, dropout=0.3)
        self.attn = nn.Linear(64,1)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(64, nc)
    def forward(self, x):
        bs, sl, nf = x.shape
        x = self.bn(x.reshape(bs*sl, nf)).reshape(bs, sl, nf)
        o,_ = self.lstm(x)
        w = torch.softmax(self.attn(o),1)
        return self.fc(self.dropout((o*w).sum(1)))

step_len = 20
output_root = PROJECT_ROOT / "artifacts" / "futures_alstm" / "alstm_l2tick_v2"
model_a = AttnLSTM(len(all_features), 2)
model_a.load_state_dict(torch.load(str(output_root / "model_a.pt"), weights_only=True))
model_a.eval()
model_b = AttnLSTM(len(all_features), 3)
model_b.load_state_dict(torch.load(str(output_root / "model_b.pt"), weights_only=True))
model_b.eval()
print("模型加载完成")

In [ ]:
# 生成预测（使用softmax概率，而非argmax）
train_mask = (merged['datetime'] >= '2024-07-01') & (merged['datetime'] <= '2025-06-30')
sc = StandardScaler()
sc.fit(merged.loc[train_mask, all_features].values.astype(np.float32))
X_all = sc.transform(merged[all_features].values.astype(np.float32))
def make_seq(X, sl): return np.array([X[i-sl:i] for i in range(sl, len(X))])
X_seq = make_seq(X_all, step_len)

with torch.no_grad():
    logits_a = model_a(torch.FloatTensor(X_seq))
    prob_a = torch.softmax(logits_a, dim=1).numpy()  # [no_trade, trade]
    logits_b = model_b(torch.FloatTensor(X_seq))
    prob_b = torch.softmax(logits_b, dim=1).numpy()  # [down, flat, up]

merged = merged.iloc[step_len:].reset_index(drop=True)
merged['trade_prob'] = prob_a[:, 1]  # trade概率
merged['prob_down'] = prob_b[:, 0]
merged['prob_flat'] = prob_b[:, 1]
merged['prob_up'] = prob_b[:, 2]
merged['dir_prob'] = merged['prob_up'] - merged['prob_down']  # 方向差值
print(f"预测完成: {len(merged):,} 行")
print(f"trade_prob 统计: mean={merged['trade_prob'].mean():.3f}, std={merged['trade_prob'].std():.3f}")
print(f"dir_prob 统计: mean={merged['dir_prob'].mean():.4f}, std={merged['dir_prob'].std():.4f}")

In [ ]:
# 使用与benchmark完全相同的回测逻辑
account = 1_500_000

# 与benchmark相同的阈值
trade_prob_threshold = 0.52
dir_prob_threshold = 0.005  # 更低的阈值

windows = [
    ("2025-01","2025-01-01","2025-03-31"),
    ("2025-04","2025-04-01","2025-06-30"),
    ("2025-07","2025-07-01","2025-09-30"),
    ("2025-10","2025-10-01","2025-12-31"),
]

results = []
for wid, start, end in windows:
    mask = (merged['datetime'] >= start) & (merged['datetime'] <= end)
    df_w = merged[mask]
    
    # 与benchmark相同的信号生成逻辑
    sig = pd.Series(0.0, index=df_w.index)
    trade_mask = df_w['trade_prob'] > trade_prob_threshold
    long_mask = trade_mask & (df_w['dir_prob'] > dir_prob_threshold)
    short_mask = trade_mask & (df_w['dir_prob'] < -dir_prob_threshold)
    sig[long_mask] = 1.0
    sig[short_mask] = -1.0
    
    if sig.abs().sum() == 0:
        print(f"{wid}: 无信号")
        results.append({'window': wid, 'return': 0, 'drawdown': 0, 'trades': 0})
        continue
    
    # 与benchmark完全相同的回测规则
    rule = StrategyRuleConfig(
        instrument="IF", strategy_mode="long_short",
        contract_multiplier=300, fixed_trade_amount=1,
        first_trade_time=pd.Timestamp("10:00").time(),
        last_trade_time=pd.Timestamp("15:00").time(),
        trade_interval_minutes=1,
        force_flatten_exec_time=pd.Timestamp("15:00").time(),
        force_flatten_daily=True, open_cost=0.0002, close_cost=0.0002,
    )
    
    price_ser = pd.Series(df_w['vwap'].values, index=df_w['datetime'])
    signal_ser = pd.Series(sig.values, index=df_w['datetime'])
    
    report, _, indicator, _ = run_backtest(
        pred=signal_ser, test_start=start, end_time=end,
        account=account, strategy_rule=rule, price_series=price_ser,
    )
    
    final = report['account'].iloc[-1]
    ret = (final - account) / account * 100
    dd = (report['account'] / report['account'].cummax() - 1).min() * 100
    tc = indicator.get('1min', {}).get('trade_count', 0)
    print(f"{wid}: 收益={ret:+.2f}%, 回撤={dd:.2f}%, 交易={tc}")
    results.append({'window': wid, 'return': ret, 'drawdown': dd, 'trades': tc})

# 对比
print('\n' + '='*70)
print('对比: Benchmark vs v2 (相同回测策略)')
print('='*70)
benchmark = [('2025-01',-0.30,-2.29,702),('2025-04',2.62,-1.55,348),('2025-07',0.57,-3.24,1506),('2025-10',-2.35,-4.42,596)]
print(f"{'窗口':<12} {'Benchmark收益':<15} {'v2收益':<15} {'Benchmark回撤':<15} {'v2回撤':<15}")
print('-'*72)
for w, br, bd, _ in benchmark:
    v2r = next((r['return'] for r in results if r['window'] == w), 'N/A')
    v2d = next((r['drawdown'] for r in results if r['window'] == w), 'N/A')
    v2rs = f"{v2r:>+.2f}%" if isinstance(v2r, float) else str(v2r)
    v2ds = f"{v2d:>+.2f}%" if isinstance(v2d, float) else str(v2d)
    print(f"{w:<12} {br:>+.2f}%{'':<10} {v2rs:<15} {bd:>+.2f}%{'':<10} {v2ds}")

json.dump({'benchmark': benchmark, 'v2': results}, open(output_root / 'backtest_comparison.json', 'w'), indent=2)
print(f"\n结果已保存")